<a href="https://colab.research.google.com/github/LoneWolf2026/Neural-Network-for-Nuclear-Binding-Energy-Prediction-/blob/main/Nuclear_Binding_Energy_NeuralNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#1. Find out what to do with uncertainty measurements from data set
#-Answer: Omit uncertainties for now. Add them later. Focus on getting the Neural net trained first
#2. Change loss function from BCELoss to MSELoss (Mean Square Error)(works for ReLu function)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
from torchsummary import summary
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [ ]:
data_df_2020 = pd.read_csv('/content/drive/MyDrive/AME_Dataset_2020.csv',header =None)
data_df_2020 = data_df_2020.drop([0, 1]).reset_index(drop=True) #removes neutron and proton from first two rows
data_df_2020 = data_df_2020.drop(columns=[0,4]).reset_index(drop=True)
data_df_2020.columns = range(data_df_2020.columns.size)
data_df_2020.head()

,0,1,2,3,4,5,6,7,8
0,1,1,2,13135.722895,0.000015,1112.28310,0.00020,14101.777844,0.000015
1,2,1,3,14949.810900,0.000080,2827.26540,0.00030,16049.281320,0.000080
2,1,2,3,14931.218880,0.000060,2572.68044,0.00015,16029.321970,0.000060
3,0,3,3,28667.000000,2000.000000,-2267.00000,667.00000,30775.000000,2147.000000
4,3,1,4,24621.129000,100.000000,1720.44910,25.00000,26431.867000,107.354000


In [ ]:
og_data = data_df_2020.copy()

for columns in data_df_2020.columns:
  data_df_2020[columns] = pd.to_numeric(data_df_2020[columns])
  data_df_2020[columns] = data_df_2020[columns]/abs(max(data_df_2020[columns]))
print(data_df_2020)

             0         1         2         3             4         5  \
0     0.005650  0.008475  0.006780  0.065232  7.488767e-09  0.126474   
1     0.011299  0.008475  0.010169  0.074241  3.994009e-08  0.321479   
2     0.005650  0.016949  0.010169  0.074149  2.995507e-08  0.292531   
3     0.000000  0.025424  0.010169  0.142361  9.985022e-01 -0.257773   
4     0.016949  0.008475  0.013559  0.122269  4.992511e-02  0.195627   
...        ...       ...       ...       ...           ...       ...   
3551  0.994350  0.991525  0.993220  0.965531  3.884174e-01  0.806749   
3552  0.988701  1.000000  0.993220  0.987252  3.539690e-01  0.804816   
3553  1.000000  0.991525  0.996610  0.975309  2.960559e-01  0.806408   
3554  0.994350  1.000000  0.996610  0.989825  2.760859e-01  0.804930   
3555  1.000000  1.000000  1.000000  1.000000  3.270095e-01  0.804589   

                 6         7             8  
0     2.998501e-07  0.014102  6.976744e-09  
1     4.497751e-07  0.016050  3.720930e-08  


In [ ]:
#Find out what to do with the error calculations listed for each data point

input = np.array(data_df_2020.iloc[:,[0,1,2,3,4,7,8]].values)
output = np.array(data_df_2020.iloc[:,[5]].values) #We start with Binding Energy, then we will decide if element name and uncertainty can be incorporated later

#print(input)
#print(output)

[[5.64971751e-03 8.47457627e-03 6.77966102e-03 ... 7.48876685e-09
  1.41020422e-02 6.97674419e-09]
 [1.12994350e-02 8.47457627e-03 1.01694915e-02 ... 3.99400899e-08
  1.60495822e-02 3.72093023e-08]
 [5.64971751e-03 1.69491525e-02 1.01694915e-02 ... 2.99550674e-08
  1.60296225e-02 2.79069767e-08]
 ...
 [1.00000000e+00 9.91525424e-01 9.96610169e-01 ... 2.96055916e-01
  2.10843953e-01 2.96279070e-01]
 [9.94350282e-01 1.00000000e+00 9.96610169e-01 ... 2.76085871e-01
  2.13983012e-01 2.76279070e-01]
 [1.00000000e+00 1.00000000e+00 1.00000000e+00 ... 3.27009486e-01
  2.16182053e-01 3.26976744e-01]]
[[0.12647406]
 [0.32147906]
 [0.29253104]
 ...
 [0.80640801]
 [0.80492982]
 [0.8045887 ]]


In [ ]:
input_train,input_test,output_train,output_test = train_test_split(input,output,test_size = 0.3)
input_test,input_val,output_test,output_val = train_test_split(input_test,output_test,test_size = 0.5)

In [ ]:
class dataset(Dataset):
  def __init__(self,input,output):
    self.input = torch.tensor(input, dtype = torch.float32).to(device)
    self.output = torch.tensor(output, dtype = torch.float32).to(device)

  def __len__(self):
    return len(self.input)

  def __getitem__(self,index):
    return self.input[index], self.output[index]

In [ ]:
training_data = dataset(input_train,output_train)
validation_data = dataset(input_val,output_val)
testing_data = dataset(input_test,output_test)

In [ ]:
train_dataloader = DataLoader(training_data,batch_size = 4,shuffle = True)
val_dataloader = DataLoader(validation_data,batch_size = 4,shuffle = True)
test_dataloader = DataLoader(testing_data,batch_size = 4,shuffle = True)

In [ ]:
for i,b in train_dataloader:
  print(i)
  print("===========")
  print(b)
  break

tensor([[ 2.6554e-01,  3.1356e-01,  2.8475e-01, -3.9608e-01,  1.0954e-03,
          9.1439e-01,  1.0953e-03],
        [ 2.6554e-01,  2.6271e-01,  2.6441e-01, -3.1635e-01,  5.2471e-04,
          9.3163e-01,  5.2419e-04],
        [ 8.9831e-01,  9.1525e-01,  9.0508e-01,  6.0912e-01,  4.7429e-02,
          1.3168e-01,  4.7442e-02],
        [ 2.0904e-01,  2.5424e-01,  2.2712e-01, -3.3709e-01,  3.7693e-04,
          9.2714e-01,  3.7674e-04]], device='cuda:0')
tensor([[0.9865],
        [0.9753],
        [0.8295],
        [0.9931]], device='cuda:0')


In [ ]:
Hidden_Neurons = 8
class BindingE_NeuralNet(nn.Module):
  def __init__(self):
    super(BindingE_NeuralNet,self).__init__()

    self.layer1 = nn.Linear(input_train.shape[1],Hidden_Neurons)
    self.OutLayer = nn.Linear(Hidden_Neurons,1) #We want at least one output: Binding Energy (uncertainty could be added later)
    self.Relu = nn.ReLU()

  def forward(self,X):

    X = self.layer1(X)
    X = self.OutLayer(X)
    X = self.Relu(X)
    #Return to this Model and refine architecture

    return X


BE_NeuralNet = BindingE_NeuralNet().to('cuda')

In [ ]:
summary(BE_NeuralNet,(input.shape[1],))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                    [-1, 8]              64
            Linear-2                    [-1, 1]               9
              ReLU-3                    [-1, 1]               0
Total params: 73
Trainable params: 73
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00
----------------------------------------------------------------


In [ ]:
criterion = nn.BCELoss()
optimizer = Adam(BE_NeuralNet.parameters(), lr = 1e-3)

In [ ]:
for data in train_dataloader:
  inputs, labels = data
  print(inputs)
  print(labels)
  break

tensor([[ 6.3277e-01,  6.1017e-01,  6.2373e-01, -2.0609e-01,  1.9823e-02,
          9.5547e-01,  1.9826e-02],
        [ 5.5932e-01,  5.6780e-01,  5.6271e-01, -3.1321e-01,  3.9241e-04,
          9.3231e-01,  3.9256e-04],
        [ 1.6384e-01,  2.4576e-01,  1.9661e-01, -2.5658e-01,  2.8158e-04,
          9.4455e-01,  2.8093e-04],
        [ 7.2881e-01,  7.7119e-01,  7.4576e-01,  1.0070e-01,  7.3165e-03,
          2.1770e-02,  7.3172e-03]], device='cuda:0')
tensor([[0.9086],
        [0.9251],
        [0.9746],
        [0.8705]], device='cuda:0')


In [ ]:
total_loss_train_plot = []
total_loss_validation_plot = []
total_acc_train_plot = []
total_acc_validation_plot = []

epochs = 10
for epoch in range(epochs):
  total_acc_train = 0
  total_loss_train = 0
  total_acc_validation = 0
  total_loss_validation = 0

  for data in train_dataloader:
    inputs, labels = data

    prediction = BE_NeuralNet(inputs)

    batch_loss = criterion(prediction,labels)
    total_loss_train += batch_loss.item()

    acc = ((prediction).round() == labels).sum().item()

    total_acc_train += acc

    batch_loss.backward()
    optimizer.step()
    optimizer.zero_grad()

  with torch.no_grad():
    for data in val_dataloader:
      inputs, labels = data

      prediction = BE_NeuralNet(inputs)
      batch_loss = criterion(prediction,labels)

      total_loss_validation += batch_loss.item()
      acc = ((prediction).round() == labels).sum().item()

      total_acc_validation += acc

  total_loss_train_plot.append(round(total_loss_train/1000,4))
  total_loss_validation_plot.append(round(total_loss_validation/1000,4))

  total_acc_train_plot.append(round(total_acc_train/training_data.__len__()*100,4))
  total_acc_validation_plot.append(round(total_acc_validation/validation_data.__len__()*100,4))

  print(f'''Epoch: {epoch+1} Train_Loss: {round(total_loss_train/1000,4)} Train_Accuracy: {round(total_acc_train/training_data.__len__()*100,4)}
        Validation Loss: {round(total_loss_validation/1000,4)} Validation_Accuracy: {round(total_acc_validation/validation_data.__len__()*100,4)}''')
  print("=="*25)

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
